In [2]:
# ================================================================
# Cr3+ Polyhedron and Structural Descriptor Extractor
# ================================================================

import warnings
import numpy as np
import pandas as pd
from scipy.spatial import ConvexHull, QhullError
from tqdm import tqdm
from pymatgen.analysis.local_env import CrystalNN
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.ext.matproj import MPRester

warnings.filterwarnings("ignore")

cnn = CrystalNN(cation_anion=True)

CATIONS = [
    "Al", "Ga", "In", "Sc", "Y", "Lu", "Gd", "La",
    "Mg", "Ti", "Zr", "Hf", "Nb", "Ta", "Sn", "Sb", "Ge", "Si",
]

COLUMNS = [
    "Formula", "Central cation", "poly vol", "Coordination No",
    "Cation ionic radii (R)", "1/R2", "R5",
    "min_metal_ligand_bond_length", "max_metal_ligand_bond_length",
    "mean_metal_ligand_bond_length", "distortion index",
    "SGR", "Volume", "Vol/fu", "c", "vol/atom",
]

def roman(n):
    vals = [1,4,5,9,10,40,50,90,100,400,500,900,1000]
    syms = ["I","IV","V","IX","X","XL","L","XC","C","CD","D","CM","M"]
    out, i = "", 12
    while n:
        d = n // vals[i]; n %= vals[i]
        out += syms[i] * d; i -= 1
    return out

def poly_volume(pts):
    try:
        return ConvexHull(pts).volume if len(set(map(tuple, pts))) >= 4 else 0
    except Exception:
        return 0

def build_record(structure, i):
    s = structure.copy()
    s.add_oxidation_state_by_guess(max_sites=-1)

    site      = s.sites[i]
    nn        = cnn.get_nn_info(s, i)
    cn        = cnn.get_cn(s, i)

    # Ionic radius & derived
    try:    r = float(site.specie.get_shannon_radius(cn=roman(int(round(cn))), radius_type="ionic"))
    except: r = float(getattr(site.specie, "average_ionic_radius", 0) or 0)
    inv_r2 = (1 / r**2) if r else None
    r5     = r**5 if r else None

    # Bond lengths
    bl = [n["site"].distance(site) for n in nn if n["site"].distance(site) > 0]
    if bl:
        mn, lo, hi = np.mean(bl), np.min(bl), np.max(bl)
        di = np.mean(np.abs(np.array(bl) - mn)) / mn
    else:
        mn = lo = hi = di = 0

    # Structure descriptors
    sga  = SpacegroupAnalyzer(structure)
    conv = sga.get_conventional_standard_structure()
    _, z = conv.composition.get_reduced_composition_and_factor()

    return {
        "Formula":                      structure.composition.reduced_formula,
        "Central cation":               str(structure.sites[i].specie),
        "poly vol":                     poly_volume([n["site"].coords for n in nn]),
        "Coordination No":              cn,
        "Cation ionic radii (R)":       r or None,
        "1/R2":                         inv_r2,
        "R5":                           r5,
        "min_metal_ligand_bond_length": lo,
        "max_metal_ligand_bond_length": hi,
        "mean_metal_ligand_bond_length":mn,
        "distortion index":             di,
        "SGR":                          sga.get_space_group_number(),
        "Volume":                       conv.volume,
        "Vol/fu":                       conv.volume / z if z else None,
        "c":                            conv.lattice.c,
        "vol/atom":                     conv.volume / conv.composition.num_atoms,
    }

# ================================================================
# Main Loop
# ================================================================

input_df       = pd.read_excel("Feature_generator.xlsx")
formula_list   = input_df["Formula"].astype(str).tolist()
records, missing = [], []

print(f"Loaded {len(formula_list)} formulas")

with MPRester("Your_MP_API") as mpr:
    for formula in formula_list:
        print(f"\nSearching: {formula}")
        try:
            docs = mpr.summary.search(formula=formula)
        except Exception as e:
            print(f"  Error: {e}"); missing.append(formula); continue

        if not docs:
            print("  Not found"); missing.append(formula); continue

        for doc in tqdm(docs, desc=formula):
            structure = doc.get("structure")
            if structure is None: continue
            try:
                sga      = SpacegroupAnalyzer(structure)
                eq_sites = set(sga.get_symmetry_dataset()["equivalent_atoms"])
            except Exception as e:
                print(f"  SGR failed: {e}"); continue

            for i in eq_sites:
                elem = str(structure.sites[i].specie)
                if not any(c in elem for c in CATIONS): continue
                cn = len(structure.get_neighbors(structure.sites[i], r=3.0))
                if cn < 4: continue
                try:
                    records.append(build_record(structure, i))
                    print(f"  + {elem} CN={cn}")
                except Exception as e:
                    print(f"  Skipped {elem}: {e}")

# ================================================================
# Save
# ================================================================

if records:
    pd.DataFrame(records)[COLUMNS].to_excel("Cr3+_polyhedron_data-1.xlsx", index=False)
    print(f"\nSaved {len(records)} records.")

if missing:
    pd.DataFrame({"Not Found": missing}).to_excel("Not_Found_on_MP.xlsx", index=False)
    print(f"Saved {len(missing)} missing formulas.")

Loaded 7 formulas

Searching: Sr2MgWO6


Sr2MgWO6: 100%|██████████| 2/2 [00:00<00:00, 32.07it/s]


  + Mg CN=6
  + Mg CN=6

Searching: TaInO4


TaInO4: 100%|██████████| 1/1 [00:00<00:00, 39.53it/s]


  + Ta CN=6
  + In CN=6

Searching: Sc2(WO4)3


...

  + Ga CN=6

Searching: Y3Al5O12


Y3Al5O12:   0%|          | 0/1 [00:00<?, ?it/s]

  + Y CN=8


Y3Al5O12: 100%|██████████| 1/1 [00:00<00:00,  6.25it/s]

  + Al CN=6
  + Al CN=4

Saved 11 records.
Saved 1 missing formulas.
